In [ ]:
import os
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Extraction Functions

def extract_text_from_txt(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def parse_resume_content(raw_text):
    """
    Heuristic-based parser to extract Name, Skills, Experience, and Education.
    Adjust regex patterns to fit your specific resume formats.
    """
    lines = [line.strip() for line in raw_text.splitlines() if line.strip()]
    name = lines[0] if lines else "Unknown"

    # Extraction patterns
    skills_match = re.search(r"(?:skills|technical skills|technologies)[:\n\s]+(.*?)(?=\n[A-Z][a-zA-Z\s]+:|\Z)", raw_text, re.IGNORECASE | re.DOTALL)
    exp_match = re.search(r"(?:experience|work experience|employment)[:\n\s]+(.*?)(?=\n[A-Z][a-zA-Z\s]+:|\Z)", raw_text, re.IGNORECASE | re.DOTALL)
    edu_match = re.search(r"(?:education|qualifications|academic)[:\n\s]+(.*?)(?=\n[A-Z][a-zA-Z\s]+:|\Z)", raw_text, re.IGNORECASE | re.DOTALL)

    return {
        "Name": name,
        "Skills": skills_match.group(1).strip() if skills_match else "N/A",
        "Experience": exp_match.group(1).strip() if exp_match else "N/A",
        "Education": edu_match.group(1).strip() if edu_match else "N/A",
        "Full_Text": raw_text
    }

In [ ]:
# 2. Ingest Multiple Resumes (TXT / CSV)
def load_resumes(folder_path_or_csv):
    candidates = []

    # If input is a single CSV file
    if os.path.isfile(folder_path_or_csv) and folder_path_or_csv.endswith(".csv"):
        df_csv = pd.read_csv(folder_path_or_csv)
        for _, row in df_csv.iterrows():
            full_text = f"{row.get('Name', '')} {row.get('Skills', '')} {row.get('Experience', '')} {row.get('Education', '')}"
            candidates.append({
                "Name": row.get("Name", "Unknown"),
                "Skills": row.get("Skills", "N/A"),
                "Experience": row.get("Experience", "N/A"),
                "Education": row.get("Education", "N/A"),
                "Full_Text": full_text
            })

    # If input is a folder containing .txt files
    elif os.path.isdir(folder_path_or_csv):
        for filename in os.listdir(folder_path_or_csv):
            if filename.endswith(".txt"):
                file_path = os.path.join(folder_path_or_csv, filename)
                raw_text = extract_text_from_txt(file_path)
                parsed_data = parse_resume_content(raw_text)
                candidates.append(parsed_data)
                
    return candidates


In [ ]:
# 3. Matching & Scoring Engine
def score_and_rank_resumes(candidates, job_description, score_threshold=10.0):
    if not candidates:
        return pd.DataFrame()

    corpus = [job_description] + [c["Full_Text"] for c in candidates]

    # Calculate TF-IDF Cosine Similarity
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(corpus)
    similarity_scores = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()

    # Assign percentage match score
    for i, candidate in enumerate(candidates):
        candidate["Match_Score (%)"] = round(similarity_scores[i] * 100, 2)

    # Convert to DataFrame, sort, and filter by threshold
    df = pd.DataFrame(candidates).drop(columns=["Full_Text"])
    df = df.sort_values(by="Match_Score (%)", ascending=False).reset_index(drop=True)
    df["Rank"] = df.index + 1

    shortlisted = df[df["Match_Score (%)"] >= score_threshold]
    return df, shortlisted

In [ ]:
# 4. Execution Pipeline
if __name__ == "__main__":
    job_description = """
    Looking for a Python Developer experienced in Django, REST APIs, SQL,
    Docker, and Machine Learning libraries like scikit-learn and pandas.
    Bachelor's degree in Computer Science or related field required.
    """

    # Example: Directory path containing .txt resumes or path to a .csv file
    RESUME_SOURCE = "/home/ddd0604/Resume.csv"
    OUTPUT_CSV_PATH = "shortlisted_candidates.csv"

    if os.path.exists(RESUME_SOURCE):
        candidate_list = load_resumes(RESUME_SOURCE)
        ranked_df, shortlisted_df = score_and_rank_resumes(
            candidate_list, 
            job_description, 
            score_threshold=10.0
        )

        # Export shortlisted candidates to CSV
        shortlisted_df.to_csv(OUTPUT_CSV_PATH, index=False)
        print(f"Exported {len(shortlisted_df)} shortlisted candidates to '{OUTPUT_CSV_PATH}'.")
    else:
        print(f"Source path '{RESUME_SOURCE}' not found. Please provide a valid directory or CSV path.")

Exported 0 shortlisted candidates to 'shortlisted_candidates.csv'.
